# Análisis epidemiológico con reglas de asociación

Este notebook trabaja con el dataset **Diabetes Health Indicators Dataset** de Kaggle/BRFSS 2015. El enfoque del proyecto es **minería de reglas de asociación**, por lo tanto el análisis es no supervisado: no se define una variable objetivo, no se entrena un clasificador y no se evalúa desempeño predictivo.

El objetivo es descubrir patrones frecuentes e interpretables entre condiciones de salud, hábitos, factores clínicos y comorbilidades mediante dos algoritmos:

- Apriori
- FP-Growth


## Definición de problemas de reglas de asociación

En este proyecto se plantean problemas exploratorios propios de reglas de asociación:

1. Identificar combinaciones frecuentes de comorbilidades asociadas a diabetes o prediabetes, como hipertensión, colesterol alto, enfermedad cardíaca, dificultad para caminar u obesidad.
2. Detectar hábitos de vida que coocurren con factores clínicos de riesgo, por ejemplo tabaquismo, baja actividad física, consumo de frutas/verduras y consumo elevado de alcohol.
3. Encontrar perfiles frecuentes de pacientes con hipertensión, colesterol alto, obesidad o enfermedades cardíacas considerando edad, salud general, salud física, salud mental, educación e ingresos.

La salida esperada no son predicciones individuales, sino reglas del tipo `antecedente -> consecuente` evaluadas con soporte, confianza y lift.


## 1. Dependencias

Se instalan paquetes faltantes solo cuando el entorno no los tiene disponibles. `mlxtend` se utiliza para generar itemsets frecuentes y reglas de asociación.


In [ ]:
import importlib
import subprocess
import sys

def install_if_missing(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])

install_if_missing('mlxtend')
install_if_missing('pandas')
install_if_missing('numpy')
install_if_missing('matplotlib')
install_if_missing('seaborn')

print('Dependencias listas')


## 2. Imports y configuración

Se cargan las librerías necesarias y se define una configuración visual básica.


In [ ]:
import glob
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)
plt.rcParams['figure.figsize'] = (9, 5)
sns.set_theme(style='whitegrid')

print('Entorno importado')


## 3. Carga del dataset

El notebook busca automáticamente un archivo CSV con la palabra `diabetes` en el directorio del proyecto. El archivo esperado es `diabetes_BRFSS2015.csv`.


In [ ]:
candidates = sorted(set(glob.glob('*diabetes*.csv') + glob.glob('*Diabetes*.csv')))
if not candidates:
    raise FileNotFoundError('No se encontró un CSV con diabetes en el nombre dentro del directorio actual.')

data_path = candidates[0]
df = pd.read_csv(data_path)

print(f'Archivo leído: {data_path}')
print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')
display(df.head())


## 4. Exploración inicial

Esta revisión verifica estructura, tipos de datos, valores nulos y duplicados. En reglas de asociación los registros repetidos pueden representar múltiples encuestados con el mismo perfil, por lo que no se eliminan automáticamente.


In [ ]:
df.info()

resumen_calidad = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'n_unicos': df.nunique()
})
display(resumen_calidad)

duplicados = int(df.duplicated().sum())
print(f'Registros duplicados exactos detectados: {duplicados:,}')
print('Se conservan porque cada fila representa una encuesta y aporta al cálculo de soporte.')


## 5. Distribuciones descriptivas

Las siguientes tablas ayudan a entender las variables que luego se convertirán en ítems. `Diabetes_012` se conserva como una condición descriptiva, no como variable objetivo.


In [ ]:
columnas_clave = ['Diabetes_012', 'HighBP', 'HighChol', 'BMI', 'Smoker', 'HeartDiseaseorAttack', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age']
display(df[columnas_clave].describe().T)

diabetes_labels = {0: 'sin_diabetes', 1: 'prediabetes', 2: 'diabetes'}
dist_diabetes = df['Diabetes_012'].map(diabetes_labels).value_counts(normalize=True).rename('proporcion').to_frame()
display(dist_diabetes)


## 6. Preparación transaccional

Para usar `mlxtend.frequent_patterns`, los datos deben estar en formato one-hot booleano: cada columna representa un ítem y cada fila una transacción.

Criterios aplicados:

- Las variables binarias se transforman solo cuando la condición está presente (`valor = 1`). No se crean ítems de ausencia como `no_fuma` o `no_diabetes`.
- `Diabetes_012` se representa únicamente como `prediabetes` o `diabetes` cuando esos estados están presentes. El valor 0 no se codifica para evitar reglas dominadas por ausencia de diabetes.
- `BMI`, `Age`, `MentHlth` y `PhysHlth` se discretizan en categorías interpretables.
- Variables ordinales como salud general, educación e ingresos se convierten en categorías.


In [ ]:
df_work = df.copy()

for col in df_work.columns:
    df_work[col] = pd.to_numeric(df_work[col], errors='coerce')

filas_antes = len(df_work)
df_work = df_work.dropna().copy()
filas_despues = len(df_work)
print(f'Filas removidas por nulos no convertibles: {filas_antes - filas_despues:,}')

binary_items = {
    'HighBP': 'hipertension',
    'HighChol': 'colesterol_alto',
    'CholCheck': 'chequeo_colesterol',
    'Smoker': 'fuma',
    'Stroke': 'derrame_cerebral',
    'HeartDiseaseorAttack': 'enfermedad_cardiaca_o_infarto',
    'PhysActivity': 'actividad_fisica',
    'Fruits': 'consume_frutas',
    'Veggies': 'consume_verduras',
    'HvyAlcoholConsump': 'consumo_alcohol_alto',
    'AnyHealthcare': 'tiene_cobertura_salud',
    'NoDocbcCost': 'no_consulto_por_costo',
    'DiffWalk': 'dificultad_caminar'
}

age_labels = {
    1: 'edad_18_24',
    2: 'edad_25_29',
    3: 'edad_30_34',
    4: 'edad_35_39',
    5: 'edad_40_44',
    6: 'edad_45_49',
    7: 'edad_50_54',
    8: 'edad_55_59',
    9: 'edad_60_64',
    10: 'edad_65_69',
    11: 'edad_70_74',
    12: 'edad_75_79',
    13: 'edad_80_mas'
}

genhlth_labels = {
    1: 'salud_general_excelente',
    2: 'salud_general_muy_buena',
    3: 'salud_general_buena',
    4: 'salud_general_regular',
    5: 'salud_general_mala'
}

education_labels = {
    1: 'educacion_sin_escolaridad',
    2: 'educacion_primaria',
    3: 'educacion_secundaria_incompleta',
    4: 'educacion_secundaria_completa',
    5: 'educacion_universitaria_incompleta',
    6: 'educacion_universitaria_completa'
}

income_labels = {
    1: 'ingreso_menor_10k',
    2: 'ingreso_10k_15k',
    3: 'ingreso_15k_20k',
    4: 'ingreso_20k_25k',
    5: 'ingreso_25k_35k',
    6: 'ingreso_35k_50k',
    7: 'ingreso_50k_75k',
    8: 'ingreso_75k_mas'
}

def add_item(items, name, condition):
    items[name] = condition.fillna(False).astype(bool)

def add_mapped_category(items, source_col, labels):
    values = df_work[source_col].round().astype(int)
    for code, label in labels.items():
        add_item(items, label, values.eq(code))

def build_transactions_onehot():
    items = {}

    add_item(items, 'prediabetes', df_work['Diabetes_012'].eq(1))
    add_item(items, 'diabetes', df_work['Diabetes_012'].eq(2))

    for source_col, item_name in binary_items.items():
        add_item(items, item_name, df_work[source_col].eq(1))

    add_item(items, 'sexo_femenino', df_work['Sex'].eq(0))
    add_item(items, 'sexo_masculino', df_work['Sex'].eq(1))

    add_item(items, 'bmi_bajo_peso', df_work['BMI'].lt(18.5))
    add_item(items, 'bmi_normal', df_work['BMI'].between(18.5, 24.9, inclusive='both'))
    add_item(items, 'bmi_sobrepeso', df_work['BMI'].between(25, 29.9, inclusive='both'))
    add_item(items, 'bmi_obesidad_i', df_work['BMI'].between(30, 34.9, inclusive='both'))
    add_item(items, 'bmi_obesidad_ii', df_work['BMI'].between(35, 39.9, inclusive='both'))
    add_item(items, 'bmi_obesidad_iii', df_work['BMI'].ge(40))

    add_item(items, 'mala_salud_mental_1_7_dias', df_work['MentHlth'].between(1, 7, inclusive='both'))
    add_item(items, 'mala_salud_mental_8_14_dias', df_work['MentHlth'].between(8, 14, inclusive='both'))
    add_item(items, 'mala_salud_mental_15_30_dias', df_work['MentHlth'].between(15, 30, inclusive='both'))
    add_item(items, 'mala_salud_fisica_1_7_dias', df_work['PhysHlth'].between(1, 7, inclusive='both'))
    add_item(items, 'mala_salud_fisica_8_14_dias', df_work['PhysHlth'].between(8, 14, inclusive='both'))
    add_item(items, 'mala_salud_fisica_15_30_dias', df_work['PhysHlth'].between(15, 30, inclusive='both'))

    add_mapped_category(items, 'Age', age_labels)
    add_mapped_category(items, 'GenHlth', genhlth_labels)
    add_mapped_category(items, 'Education', education_labels)
    add_mapped_category(items, 'Income', income_labels)

    onehot = pd.DataFrame(items, index=df_work.index).astype(bool)
    onehot = onehot.loc[:, onehot.any(axis=0)]
    return onehot

df_onehot = build_transactions_onehot()

print(f'Matriz one-hot: {df_onehot.shape[0]:,} transacciones x {df_onehot.shape[1]:,} ítems')
display(df_onehot.head())


### Ejemplo de transacciones

Cada fila del dataset se interpreta como una transacción compuesta por los ítems verdaderos para ese encuestado.


In [ ]:
transactions_preview = df_onehot.head(5).apply(lambda row: sorted(row.index[row].tolist()), axis=1)
for idx, transaction in transactions_preview.items():
    print(f'Transacción {idx}: {transaction}')


### Frecuencia individual de ítems

Antes de aplicar Apriori y FP-Growth se revisan los ítems más frecuentes. Esto ayuda a justificar un soporte mínimo que no sea ni demasiado laxo ni demasiado restrictivo.


In [ ]:
item_support = df_onehot.mean().sort_values(ascending=False).rename('support').to_frame()
display(item_support.head(20))

ax = item_support.head(15).sort_values('support').plot(kind='barh', legend=False)
ax.set_xlabel('Soporte')
ax.set_ylabel('Ítem')
ax.set_title('Ítems más frecuentes')
plt.show()


## 7. Criterios de filtrado de reglas

Se usan criterios explícitos para ambos modelos:

- `min_support = 0.03`: el patrón debe aparecer en al menos 3% de las transacciones. Con más de 250 mil registros, este umbral conserva patrones frecuentes y evita combinaciones demasiado raras.
- `min_confidence = 0.60`: se conservan reglas cuyo consecuente aparece en al menos 60% de las transacciones que contienen el antecedente.
- `lift > 1`: se retienen reglas con asociación positiva frente a la independencia estadística.
- `max_len = 4`: limita el tamaño de los itemsets para mantener reglas interpretables y tiempos de ejecución razonables.


In [ ]:
MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.60
MAX_LEN = 4

print(f'min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE}, lift > 1, max_len={MAX_LEN}')


## 8. Funciones para generar y exportar reglas

Las funciones siguientes ejecutan cada algoritmo, generan reglas con `association_rules`, filtran por `lift > 1`, ordenan por lift, confianza y soporte, y preparan una versión exportable a CSV.


In [ ]:
def itemset_to_text(itemset):
    return ', '.join(sorted(itemset))

def prepare_rules_for_export(rules_df):
    rules_export = rules_df.copy()
    rules_export['antecedents'] = rules_export['antecedents'].apply(itemset_to_text)
    rules_export['consequents'] = rules_export['consequents'].apply(itemset_to_text)
    return rules_export

def get_association_rules(frequent_itemsets):
    if frequent_itemsets.empty:
        return pd.DataFrame()

    rules_df = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
    if rules_df.empty:
        return rules_df

    rules_df = rules_df[rules_df['lift'] > 1].copy()
    rules_df = rules_df.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)
    return rules_df

def run_algorithm(algorithm_name, algorithm_func, output_csv):
    start = time.perf_counter()
    frequent_itemsets = algorithm_func(
        df_onehot,
        min_support=MIN_SUPPORT,
        use_colnames=True,
        max_len=MAX_LEN
    )
    elapsed = time.perf_counter() - start

    frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)
    rules_df = get_association_rules(frequent_itemsets)
    rules_export = prepare_rules_for_export(rules_df) if not rules_df.empty else rules_df
    rules_export.to_csv(output_csv, index=False)

    print(f'{algorithm_name}: {len(frequent_itemsets):,} itemsets frecuentes')
    print(f'{algorithm_name}: {len(rules_df):,} reglas filtradas')
    print(f'{algorithm_name}: {elapsed:.2f} segundos')
    print(f'Archivo exportado: {output_csv}')

    return frequent_itemsets, rules_df, elapsed


## 9. Modelo 1: Apriori

Apriori genera candidatos de itemsets frecuentes de forma iterativa. Se aplica sobre la matriz one-hot y luego se derivan reglas de asociación.


In [ ]:
itemsets_apriori, reglas_apriori, tiempo_apriori = run_algorithm(
    'Apriori',
    apriori,
    'reglas_apriori.csv'
)

display(prepare_rules_for_export(reglas_apriori).head(20))


## 10. Modelo 2: FP-Growth

FP-Growth evita la generación explícita de candidatos mediante una estructura FP-tree. Suele ser más eficiente que Apriori en datasets grandes.


In [ ]:
itemsets_fpgrowth, reglas_fpgrowth, tiempo_fpgrowth = run_algorithm(
    'FP-Growth',
    fpgrowth,
    'reglas_fpgrowth.csv'
)

display(prepare_rules_for_export(reglas_fpgrowth).head(20))


## 11. Comparación entre Apriori y FP-Growth

Se comparan cantidad de itemsets frecuentes, cantidad de reglas generadas, tiempo de ejecución y coincidencias entre las mejores reglas. Dado que ambos algoritmos usan los mismos datos y umbrales, deberían encontrar los mismos itemsets y reglas; la diferencia principal esperada está en el tiempo de ejecución.


In [ ]:
comparison = pd.DataFrame([
    {
        'modelo': 'Apriori',
        'itemsets_frecuentes': len(itemsets_apriori),
        'reglas_generadas': len(reglas_apriori),
        'tiempo_segundos': tiempo_apriori
    },
    {
        'modelo': 'FP-Growth',
        'itemsets_frecuentes': len(itemsets_fpgrowth),
        'reglas_generadas': len(reglas_fpgrowth),
        'tiempo_segundos': tiempo_fpgrowth
    }
])

display(comparison)

ax = comparison.set_index('modelo')['tiempo_segundos'].plot(kind='bar', color=['#4C78A8', '#F58518'])
ax.set_ylabel('Segundos')
ax.set_title('Tiempo de ejecución por algoritmo')
plt.xticks(rotation=0)
plt.show()


### Similitudes y diferencias entre las mejores reglas

La comparación se realiza con las reglas mejor posicionadas por lift, confianza y soporte. Para comparar coincidencias se usa la firma `antecedentes -> consecuentes`.


In [ ]:
def add_rule_signature(rules_df):
    rules_text = prepare_rules_for_export(rules_df)
    if rules_text.empty:
        rules_text['regla'] = []
        return rules_text
    rules_text['regla'] = rules_text['antecedents'] + ' -> ' + rules_text['consequents']
    return rules_text

top_n = 20
top_apriori = add_rule_signature(reglas_apriori).head(top_n)
top_fpgrowth = add_rule_signature(reglas_fpgrowth).head(top_n)

common_rules = sorted(set(top_apriori['regla']).intersection(set(top_fpgrowth['regla'])))
only_apriori = sorted(set(top_apriori['regla']).difference(set(top_fpgrowth['regla'])))
only_fpgrowth = sorted(set(top_fpgrowth['regla']).difference(set(top_apriori['regla'])))

print(f'Reglas comunes en el top {top_n}: {len(common_rules)}')
display(pd.DataFrame({'reglas_comunes': common_rules[:10]}))

print(f'Reglas solo en top Apriori: {len(only_apriori)}')
display(pd.DataFrame({'solo_top_apriori': only_apriori[:10]}))

print(f'Reglas solo en top FP-Growth: {len(only_fpgrowth)}')
display(pd.DataFrame({'solo_top_fpgrowth': only_fpgrowth[:10]}))


## 12. Interpretación de reglas destacadas

La tabla siguiente consolida las mejores reglas de ambos modelos. Para interpretarlas:

- **Soporte**: proporción de encuestados que contienen antecedente y consecuente.
- **Confianza**: probabilidad empírica del consecuente cuando aparece el antecedente.
- **Lift**: valores mayores a 1 indican asociación positiva respecto a la independencia.

Una regla con alto lift no implica causalidad; solo evidencia coocurrencia frecuente bajo los filtros definidos.


In [ ]:
best_rules = pd.concat([
    add_rule_signature(reglas_apriori).assign(modelo='Apriori').head(10),
    add_rule_signature(reglas_fpgrowth).assign(modelo='FP-Growth').head(10)
], ignore_index=True)

cols_to_show = ['modelo', 'regla', 'support', 'confidence', 'lift', 'leverage', 'conviction']
display(best_rules[cols_to_show])


## 13. Archivos exportados

Al ejecutar el notebook se generan dos archivos CSV separados con las reglas filtradas y ordenadas:

- `reglas_apriori.csv`
- `reglas_fpgrowth.csv`


In [ ]:
for file_name in ['reglas_apriori.csv', 'reglas_fpgrowth.csv']:
    path = Path(file_name)
    if path.exists():
        print(f'{file_name}: {path.stat().st_size:,} bytes')
    else:
        print(f'{file_name}: no encontrado')


## 14. Conclusiones

Apriori y FP-Growth permiten descubrir reglas de asociación entre condiciones de salud, hábitos, factores clínicos y características sociodemográficas del dataset BRFSS 2015. Ambos modelos trabajan sobre una representación transaccional one-hot y producen reglas interpretables evaluadas con soporte, confianza y lift.

Este proyecto no entrena un modelo predictivo tradicional. No se define una variable objetivo, no se divide el dataset en entrenamiento y prueba, y no se reportan métricas de clasificación. `Diabetes_012` se trata como un ítem descriptivo para identificar coocurrencias con diabetes o prediabetes, no como etiqueta a predecir.

El resultado principal son reglas interpretables, no predicciones individuales. Estas reglas pueden servir como insumo exploratorio para comprender perfiles frecuentes de riesgo, orientar análisis epidemiológicos posteriores o alimentar futuras hipótesis y modelos supervisados. Sin embargo, no reemplazan un modelo de clasificación ni permiten afirmar causalidad.
